# ASSIGNMENT 11
## Fine-Tuning GPT-2 on a Custom Style

**Name:** **Prakruthi BR**

**Reg NO:** **2411021240038**

**Subject:** **Generative AI**
### Objective
Fine-tune the pretrained GPT-2 model on a custom dataset containing approximately 200–300 lines of text with a consistent writing style. Then use the same prompt from Assignment 2 with both the original and fine-tuned GPT-2 models and compare their generated outputs.

### Assignment 2 prompt
`The projects focuses on solving a real-world problem using technology (AI).`

The same prompt is used for both models to make the comparison fair.


## 1. Install Required Libraries
The Hugging Face Transformers and Datasets libraries are used to load GPT-2, tokenize the dataset, and perform fine-tuning.

In [ ]:
!pip install -q transformers datasets accelerate

## 2. Import Libraries
PyTorch is used for model execution, while Hugging Face Transformers provides GPT-2 and the Trainer API.

In [ ]:
import os
import re
import random
import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cpu
CUDA available: False


## 3. Upload the Custom Dataset
Upload `custom_style.txt` from this ZIP folder to Google Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving custom_style.txt to custom_style (1).txt


## 4. Load the Custom Dataset
The dataset contains 250 lines of consistent fictional conversational writing.

In [ ]:
file_path = "custom_style.txt"

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

lines = raw_text.splitlines()

print("Number of lines:", len(lines))
print("\nFirst 20 lines:")
for i, line in enumerate(lines[:20], 1):
    print(f"{i}: {line}")


Number of lines: 250

First 20 lines:
1: Are you ready to leave?
2: I'm just trying to make the right choice.
3: Do you think there is a right choice today?
4: Because I am afraid of what comes next tonight.
5: Maybe tomorrow will make the answer clearer.
6: Different does not always mean worse.
7: I would take one small step.
8: And if that step went wrong today?
9: It is not simple tonight. It is just possible tonight.
10: Maybe tomorrow will make the answer clearer.
11: Why did you come here tonight?
12: Home did not feel quiet anymore.
13: Is something bothering you today?
14: Not yet tonight.
15: Maybe tomorrow will make the answer clearer.
16: You always know what to say.
17: The lights across the street.
18: They look peaceful from here.
19: Sometimes, but I would not go back tonight.
20: Maybe tomorrow will make the answer clearer.


## 5. Clean the Dataset
Empty lines, common timestamps, username prefixes, and excessive spaces are removed so the model learns language rather than irrelevant metadata.

In [ ]:
def clean_line(line):
    line = line.strip()
    line = re.sub(r'^\s*\[?\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM)?\]?\s*', '', line, flags=re.I)
    line = re.sub(r'^\s*(?:User|Username|Name)\s*[:\-]\s*', '', line, flags=re.I)
    line = re.sub(r'\s+', ' ', line)
    return line.strip()

cleaned_lines = [clean_line(line) for line in lines]
cleaned_lines = [line for line in cleaned_lines if line]

print("Original lines:", len(lines))
print("Cleaned lines:", len(cleaned_lines))


Original lines: 250
Cleaned lines: 250


## 6. Display the Cleaned Dataset
The first lines are displayed to verify that the dataset contains only the intended writing style.

In [ ]:
for i, line in enumerate(cleaned_lines[:20], 1):
    print(f"{i}: {line}")


1: Are you ready to leave?
2: I'm just trying to make the right choice.
3: Do you think there is a right choice today?
4: Because I am afraid of what comes next tonight.
5: Maybe tomorrow will make the answer clearer.
6: Different does not always mean worse.
7: I would take one small step.
8: And if that step went wrong today?
9: It is not simple tonight. It is just possible tonight.
10: Maybe tomorrow will make the answer clearer.
11: Why did you come here tonight?
12: Home did not feel quiet anymore.
13: Is something bothering you today?
14: Not yet tonight.
15: Maybe tomorrow will make the answer clearer.
16: You always know what to say.
17: The lights across the street.
18: They look peaceful from here.
19: Sometimes, but I would not go back tonight.
20: Maybe tomorrow will make the answer clearer.


In [ ]:
with open("custom_style_cleaned.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print("Cleaned dataset saved.")


Cleaned dataset saved.


## 7. Convert the Dataset to a Hugging Face Dataset

In [ ]:
dataset = Dataset.from_dict({"text": cleaned_lines})
print(dataset)


Dataset({
    features: ['text'],
    num_rows: 250
})


## 8. Load the Pretrained GPT-2 Model
The same `gpt2` model family used in Assignment 2 is loaded as the base model.

In [ ]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print("GPT-2 loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully.
Vocabulary size: 50257


## 9. Tokenize the Dataset
GPT-2 requires numerical token IDs. The tokenizer converts each text example into tokens. A maximum length of 128 tokens is used.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 250
})


## 10. Prepare the Data Collator
GPT-2 is a causal language model, so masked language modeling is disabled with `mlm=False`.

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator ready.")


Data collator ready.


## 11. Set a Random Seed
A fixed seed makes the experiment more reproducible.

In [ ]:
set_seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


## 12. Configure Fine-Tuning
The dataset is small, so three epochs and a small batch size are used. The model checkpoints are saved after each epoch.

In [ ]:
output_dir = "./gpt2_custom_style"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer configured.")

Trainer configured.


## 13. Fine-Tune GPT-2
This is the main training step. The pretrained GPT-2 parameters are adapted using the custom writing-style dataset.

In [ ]:
train_result = trainer.train()


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.166121
20,2.314735
30,2.250442
40,1.725095
50,1.635349
60,1.479345
70,1.358457
80,1.043152
90,1.004478
100,0.978638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 14. Save the Fine-Tuned Model

In [ ]:
final_model_path = "./gpt2_custom_style_final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("Fine-tuned model saved to:", final_model_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: ./gpt2_custom_style_final


## 15. Assignment 2 Prompt
The exact main prompt from Assignment 2 is used for both models:

**The projects focuses on solving a real-world problem using technology (AI).**

Using the same prompt makes the before-and-after comparison fair.

In [ ]:
prompt = "The projects focuses on solving a real-world problem using technology (AI)."
print(prompt)


The projects focuses on solving a real-world problem using technology (AI).


## 16. Generate Text with the Original GPT-2
The original pretrained model is used without fine-tuning to produce the baseline output.

In [ ]:
original_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
original_model = GPT2LMHeadModel.from_pretrained("gpt2")

original_tokenizer.pad_token = original_tokenizer.eos_token
original_model.config.pad_token_id = original_tokenizer.pad_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
original_model.to(device)

set_seed(42)

inputs = original_tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    output = original_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=original_tokenizer.eos_token_id
    )

original_output = original_tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("===== ORIGINAL GPT-2 OUTPUT =====")
print(original_output)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

===== ORIGINAL GPT-2 OUTPUT =====
The projects focuses on solving a real-world problem using technology (AI). The project was a collaboration between Google and the California Institute of Technology (Caltech), a nonprofit nonprofit organization. The team is led by John D. Schulte, who will lead the project.

The team will use a variety of approaches and tools to build out the network. The project will take a traditional computer model and connect it with a network of 3D models. The system will use the data collected to create a network of nodes on the network, which is similar to a computer


In [ ]:
with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write("Prompt:\n" + prompt + "\n\n")
    f.write("Original GPT-2 Output:\n" + original_output)

print("Saved base_model_output.txt")


Saved base_model_output.txt


## 17. Generate Text with the Fine-Tuned GPT-2
The fine-tuned model receives exactly the same prompt and generation settings.

In [ ]:
finetuned_tokenizer = GPT2Tokenizer.from_pretrained(final_model_path)
finetuned_model = GPT2LMHeadModel.from_pretrained(final_model_path)

finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
finetuned_model.config.pad_token_id = finetuned_tokenizer.pad_token_id
finetuned_model.to(device)

set_seed(42)

inputs = finetuned_tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    output = finetuned_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=finetuned_tokenizer.eos_token_id
    )

finetuned_output = finetuned_tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("===== FINE-TUNED GPT-2 OUTPUT =====")
print(finetuned_output)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

===== FINE-TUNED GPT-2 OUTPUT =====
The projects focuses on solving a real-world problem using technology (AI). But there are a few ways to make the same choice today.

Today, I want to make the same choice. I want to make the same choice tonight. I want to make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I will make the same choice tonight. I


In [ ]:
with open("finetuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write("Prompt:\n" + prompt + "\n\n")
    f.write("Fine-Tuned GPT-2 Output:\n" + finetuned_output)

print("Saved finetuned_model_output.txt")


Saved finetuned_model_output.txt


## 18. Before-and-After Comparison
The two models received the same prompt. Compare their vocabulary, tone, sentence structure, conversational patterns, and repetition.

In [ ]:
print("PROMPT")
print("" * 80)
print(prompt)


print("\nORIGINAL GPT-2")
print("" * 80)
print(original_output)


print("\nFINE-TUNED GPT-2")
print("" * 80)
print(finetuned_output)


PROMPT

The projects focuses on solving a real-world problem using technology (AI).

ORIGINAL GPT-2

The projects focuses on solving a real-world problem using technology (AI). The project was a collaboration between Google and the California Institute of Technology (Caltech), a nonprofit nonprofit organization. The team is led by John D. Schulte, who will lead the project.

The team will use a variety of approaches and tools to build out the network. The project will take a traditional computer model and connect it with a network of 3D models. The system will use the data collected to create a network of nodes on the network, which is similar to a computer

FINE-TUNED GPT-2

The projects focuses on solving a real-world problem using technology (AI). But there are a few ways to make the same choice today.

Today, I want to make the same choice. I want to make the same choice tonight. I want to make the same choice tonight. I will make the same choice tonight. I will make the same choic

## 19. Analysis of Style Differences

After observing the actual outputs above, identify at least two clear differences.

### Difference 1 — Vocabulary
Describe whether the fine-tuned model uses vocabulary or expressions that are more similar to the custom dataset.

### Difference 2 — Tone
Describe whether the fine-tuned model has a more conversational, emotional, formal, dramatic, or other tone matching the dataset.

### Additional observations
You may also discuss sentence length, sentence structure, dialogue patterns, repeated expressions, or other stylistic characteristics.

**Important:** The final statements should be based on the actual outputs produced by the notebook, not assumed results.


## 20. Conclusion

The pretrained GPT-2 model was fine-tuned on a custom dataset containing approximately 200–300 lines of consistent writing. The same Assignment 2 prompt was then given to both the original and fine-tuned models. The generated outputs were compared to determine whether fine-tuning influenced the model's vocabulary, tone, sentence structure, and overall writing style.

The experiment demonstrates how fine-tuning can adapt a pretrained language model toward a specific writing style. Because the custom dataset is relatively small, the fine-tuned model may still retain general GPT-2 behavior and may sometimes repeat patterns from the training data.
